# Giggsdance on Kaggle — 60 fps video with MiniMax H3

Generate a clip with [MiniMax H3](https://huggingface.co/MiniMaxAI/MiniMax-H3), convert it to a true 60 fps, optionally upscale, and encode a 10-bit master.

## Two things to set before you run anything

1. **Settings → Internet: ON.** Off by default on Kaggle. Without it nothing can reach PyPI or Modal and every cell below fails.
2. **Settings → Accelerator: None.** Deliberate — see below. Leaving a GPU attached just burns your weekly quota for nothing.

## Why Kaggle's GPU is not used

H3 needs about **124 GB** of weights in bf16 (61.7 GB transformer + 62.1 GB Qwen3-VL conditioner). Kaggle gives you a T4 or P100 with **16 GB** of VRAM, roughly **30 GB** of system RAM, and a disk quota well under the ~90 GB checkpoint. Even the aggressive int8 + block-streaming path needs ~75 GB of *host* RAM, so it does not fit either. Kaggle cannot run this model locally, and no amount of quantisation changes that.

So this notebook uses Kaggle as a **client**: the code runs here, the GPU is Modal's (a B200, billed per second). That is why the accelerator should be None.

## Licence — read before running

The [MiniMax H3 licence](https://github.com/Hvkki/minimax/blob/main/NOTICE.md) grants **no rights in the EU, UK, South Korea or USA**, and the restriction covers the model's **outputs**, not just its weights. If you publish a result, mark it as AI-generated.

## 1. Install

The `!` prefix runs a shell command. Without it the cell is parsed as Python and you get `SyntaxError: invalid syntax`.

In [ ]:
!git clone -q https://github.com/Hvkki/minimax.git /kaggle/working/minimax
%cd /kaggle/working/minimax
!pip install -q modal
print("installed")

## 2. Credentials

Get a token pair from [modal.com/settings/tokens](https://modal.com/settings/tokens).

Store them in **Add-ons → Secrets** as `MODAL_TOKEN_ID` and `MODAL_TOKEN_SECRET`, then run the cell below. Use Secrets rather than typing the tokens into a cell — a public Kaggle notebook shows its source, and saved output can leak them.

In [ ]:
import notebook

notebook.load_kaggle_secrets()   # reads Add-ons -> Secrets
notebook.check_setup()

If you would rather not use Secrets, set them directly instead — but do not save the notebook publicly afterwards:

```python
import os
os.environ["MODAL_TOKEN_ID"] = "ak-..."
os.environ["MODAL_TOKEN_SECRET"] = "as-..."
```

## 3. Free check — no GPU, no Modal account, $0

Runs the real interpolation, geometry and encoding stages on synthetic frames, asserting frame count, fps, bit depth, colour tags and A/V sync. Do this before spending anything.

In [ ]:
notebook.dry_run()

## 4. Measure the cold start (optional, recommended once)

Loading ~124 GB happens before a single frame is generated, and it is the least predictable cost in the system. Measuring it once makes every later budget reliable.

**The first run also downloads ~90 GB of weights into a Modal Volume.** That happens on cheap CPU inside Modal — not on Kaggle, and not against your Kaggle disk — and is skipped on every later run.

In [ ]:
notebook.probe()

## 5. Render

Defaults are the cheap ones: 5 s, `native` resolution (**no** super-resolution — it measured at ~84% of post-processing time), 60 fps, 8 steps.

`budget_usd` becomes a hard container timeout, so an overrun is killed rather than billed. Output is written to `/kaggle/working`, which is the only place Kaggle surfaces in the Output tab.

In [ ]:
path, report = notebook.render(
    prompt="a paper boat drifting down a rain-filled gutter at night, neon reflections",
    duration_s=5.0,
    resolution="native",   # "1080p" / "1440p" / "2160p" to enable upscaling
    fps=60,
    steps=8,
    budget_usd=1.00,
)
print(path)

## 6. Variations

H3's real constraints: clips are **5–14.375 s** (frame counts must be `17n+5`), generation is always **24 fps**, and the 16:9 canvas is actually **1344x768**.

```python
notebook.render(duration_s=10, steps=16)                     # longer, better
notebook.render(resolution="1440p", steps=24, budget_usd=3)  # upscaling on
notebook.render(aspect_ratio="9:16", resolution="1080p")     # vertical
notebook.render(seed=42)                                     # reproducible
```

Prompt structure matters. H3's Context-IR module is closed-source, so a structured prompt is a large step up from a one-liner:

```python
from giggsdance import Storyboard, Shot

board = Storyboard(
    style="Cinematic 2D vector animation, flat colour",
    shots=[
        Shot(description="A stick figure sprints across a white void.", camera="Static wide"),
        Shot(description="It lands a spinning kick; ripples spread outward.", start_s=3.0),
    ],
    soundscape="Sharp whooshes, a percussive thud on impact, faint room tone.",
    music="Driving synth arpeggio building to a hit on the kick.",
)
notebook.render(prompt=board.render())
```

## 7. Download

Rendered files land in `/kaggle/working` and appear in the **Output** tab on the right. Kaggle wipes everything else when the session ends.

In [ ]:
!ls -lh /kaggle/working/*.mp4 2>/dev/null || echo "nothing rendered yet"

## Troubleshooting

| Symptom | Cause |
|---|---|
| `SyntaxError: invalid syntax` on `git clone` | shell command in a Python cell — prefix with `!` |
| pip or Modal cannot connect | **Settings → Internet is OFF** |
| `Bad Request: Unsupported URL` | pasting a repo URL into Modal; Modal uploads local files instead |
| `ModuleNotFoundError: giggsdance` | wrong directory — `%cd /kaggle/working/minimax` |
| secrets not found | Add-ons → Secrets, named exactly `MODAL_TOKEN_ID` / `MODAL_TOKEN_SECRET` |
| timeout | raise `budget_usd`, or lower `steps` |
| generation error | `notebook.describe()` prints the real pipeline signature |
| output missing | it must be under `/kaggle/working` to appear in Output |

`stages/generate.py` is the one part of this repo its author never executed — it needs ~124 GB of weights and a large GPU. If something breaks, start there.